# ArUco net value inference

Per-frame grid125 ArUco scoring for picklist videos:
- IDs **0–119** → pick (`+1` each)
- IDs **990–994** → place (`-1` each)

Change `VIDEO_NAME` and `FRAME_STRIDE` in the config cell, then run all cells.

In [1]:
import csv
import sys
from pathlib import Path

import cv2
import numpy as np

# Repo root (for aruco_shelf_markers constants)
REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.aruco_shelf_markers import VALID_SHELF_MARKERS_GRID125

# ---- video pipeline: change VIDEO_NAME to run on another picklist ----
VIDEO_DIR = Path("../hmm-testing/picklist_videos")
VIDEO_NAME = "picklist_191.MP4"
VIDEO_PATH = VIDEO_DIR / VIDEO_NAME

# ---- output paths ----
CSV_OUTPUT_DIR = Path("csv_outputs")
CSV_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = CSV_OUTPUT_DIR / f"{VIDEO_PATH.stem}_aruco_net_timeseries.csv"
ANNOTATED_VIDEO = f"{VIDEO_PATH.stem}_aruco_net_annotated.mp4"

# ---- inference knobs ----
FRAME_STRIDE = 5  # score every Nth frame (raise to speed up)

# Match extract_features.py preprocessing
ORIGINAL_FRAME_WIDTH = 1920
ORIGINAL_FRAME_HEIGHT = 1080
ARUCO_DICT = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_5X5_1000)

PLACE_MARKER_IDS = frozenset({990, 991, 992, 993, 994})
PICK_MARKER_IDS = frozenset(range(120))

print("video:", VIDEO_PATH)
print("csv:", OUTPUT_CSV)
print("annotated:", ANNOTATED_VIDEO)
print("FRAME_STRIDE:", FRAME_STRIDE)
print("grid125 markers:", len(VALID_SHELF_MARKERS_GRID125))

video: ../hmm-testing/picklist_videos/picklist_191.MP4
csv: csv_outputs/picklist_191_aruco_net_timeseries.csv
annotated: picklist_191_aruco_net_annotated.mp4
FRAME_STRIDE: 5
grid125 markers: 125


In [2]:
_aruco_detector = cv2.aruco.ArucoDetector(ARUCO_DICT)


def preprocess_frame_bgr(frame_bgr):
    """Resize + RGB, matching extract_features.py."""
    resized = cv2.resize(frame_bgr, (ORIGINAL_FRAME_WIDTH, ORIGINAL_FRAME_HEIGHT))
    return cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)


def marker_value(marker_id: int) -> int:
    """Grid125 convention: 0-119 pick (+1), 990-994 place (-1)."""
    if marker_id in PICK_MARKER_IDS:
        return 1
    if marker_id in PLACE_MARKER_IDS:
        return -1
    return 0


def detect_grid125_markers(frame_bgr):
    """
    Detect ArUco markers on one frame.
    Returns list of dicts: {id, corners (4x2 in original frame coords), center (x,y)}.
    Only includes grid125 shelf IDs (0-119, 990-994).
    """
    rgb = preprocess_frame_bgr(frame_bgr)
    corners, ids, _ = _aruco_detector.detectMarkers(rgb)
    if ids is None or len(ids) == 0:
        return []

    h_orig, w_orig = frame_bgr.shape[:2]
    sx = w_orig / ORIGINAL_FRAME_WIDTH
    sy = h_orig / ORIGINAL_FRAME_HEIGHT

    detections = []
    for i, marker_id in enumerate(ids.flatten().astype(int)):
        if marker_id not in VALID_SHELF_MARKERS_GRID125:
            continue
        pts = corners[i][0].astype(np.float32)
        pts_orig = pts.copy()
        pts_orig[:, 0] *= sx
        pts_orig[:, 1] *= sy
        cx = float(np.mean(pts_orig[:, 0]))
        cy = float(np.mean(pts_orig[:, 1]))
        detections.append(
            {
                "id": marker_id,
                "corners": pts_orig,
                "center": (cx, cy),
            }
        )
    return detections


def score_frame(detections):
    """Deduplicate marker IDs and compute net pick/place score."""
    unique_ids = sorted({d["id"] for d in detections})
    pick_count = sum(1 for mid in unique_ids if mid in PICK_MARKER_IDS)
    place_count = sum(1 for mid in unique_ids if mid in PLACE_MARKER_IDS)
    net_aruco = sum(marker_value(mid) for mid in unique_ids)
    return net_aruco, pick_count, place_count, unique_ids


def analyze_frame(frame_bgr):
    """One frame -> (net_aruco, pick_count, place_count, unique_ids, detections)."""
    detections = detect_grid125_markers(frame_bgr)
    net_aruco, pick_count, place_count, unique_ids = score_frame(detections)
    return net_aruco, pick_count, place_count, unique_ids, detections

In [3]:
def run_on_video(video_path, output_csv):
    """Sample every FRAME_STRIDE frames; write net ArUco CSV."""
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0

    times, frame_indices = [], []
    net_vals, pick_counts, place_counts = [], [], []
    detected_id_strs = []
    frame_detections = {}  # frame_idx -> list of detection dicts (for annotation)

    frame_idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if frame_idx % FRAME_STRIDE == 0:
            net_aruco, pick_count, place_count, unique_ids, detections = analyze_frame(frame)
            times.append(frame_idx / fps)
            frame_indices.append(frame_idx)
            net_vals.append(net_aruco)
            pick_counts.append(pick_count)
            place_counts.append(place_count)
            detected_id_strs.append(";".join(str(i) for i in unique_ids))
            frame_detections[frame_idx] = detections
        frame_idx += 1
    cap.release()

    with open(output_csv, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["time_s", "frame_idx", "net_aruco", "pick_count", "place_count", "detected_ids"])
        for t, fi, net_v, pc, plc, ids_s in zip(
            times, frame_indices, net_vals, pick_counts, place_counts, detected_id_strs
        ):
            w.writerow([f"{t:.3f}", fi, net_v, pc, plc, ids_s])

    print(f"{video_path} -> {output_csv}  ({len(times)} rows)")
    return frame_detections


frame_detections = run_on_video(VIDEO_PATH, OUTPUT_CSV)

../hmm-testing/picklist_videos/picklist_191.MP4 -> csv_outputs/picklist_191_aruco_net_timeseries.csv  (459 rows)


In [4]:
# --- annotated video (full frame rate; overlay from most recent sampled row) ---

def load_net_csv(csv_path):
    """Returns list of (frame_idx, time_s, net_aruco, pick_count, place_count, detected_ids)."""
    rows = []
    with open(csv_path, newline="") as f:
        reader = csv.reader(f)
        next(reader, None)
        for row in reader:
            rows.append(
                (
                    int(row[1]),
                    float(row[0]),
                    int(row[2]),
                    int(row[3]),
                    int(row[4]),
                    row[5],
                )
            )
    return rows


def row_for_frame(frame_idx, rows):
    """Most recent scored row at or before this video frame."""
    if not rows:
        return None
    chosen = rows[0]
    for row in rows:
        if row[0] <= frame_idx:
            chosen = row
        else:
            break
    return chosen


def draw_stacked_overlay(frame_bgr, lines, margin=16):
    """Stack multiple text lines in a single black box, lower-right."""
    h, w = frame_bgr.shape[:2]
    font = cv2.FONT_HERSHEY_SIMPLEX
    scale = max(0.9, min(w, h) / 900.0)
    thickness = max(2, int(round(scale * 2)))
    pad = 10
    line_gap = 6

    sizes = [cv2.getTextSize(ln, font, scale, thickness) for ln in lines]
    box_w = max(tw for (tw, _), _ in sizes) + 2 * pad
    box_h = sum(th + baseline for (_, th), baseline in sizes) + line_gap * (len(lines) - 1) + 2 * pad

    x2, y2 = w - margin, h - margin
    x1, y1 = x2 - box_w, y2 - box_h
    cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), (0, 0, 0), -1)

    cursor_y = y1 + pad
    for ln, ((tw, th), baseline) in zip(lines, sizes):
        cursor_y += th
        cv2.putText(
            frame_bgr,
            ln,
            (x1 + pad, cursor_y),
            font,
            scale,
            (255, 255, 255),
            thickness,
            cv2.LINE_AA,
        )
        cursor_y += baseline + line_gap


def draw_marker_overlays(frame_bgr, detections):
    """Draw detected marker polygons and IDs on the frame."""
    for det in detections:
        mid = det["id"]
        color = (0, 255, 0) if mid in PICK_MARKER_IDS else (0, 0, 255)
        pts = det["corners"].astype(np.int32).reshape((-1, 1, 2))
        cv2.polylines(frame_bgr, [pts], True, color, 2)
        cx, cy = det["center"]
        cv2.putText(
            frame_bgr,
            str(mid),
            (int(cx), int(cy)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            color,
            2,
            cv2.LINE_AA,
        )


net_rows = load_net_csv(OUTPUT_CSV)

cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise FileNotFoundError(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
fw = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
fh = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(ANNOTATED_VIDEO, fourcc, fps, (fw, fh))
if not writer.isOpened():
    cap.release()
    raise RuntimeError(f"Could not open writer for {ANNOTATED_VIDEO}")

frame_idx = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break

    row = row_for_frame(frame_idx, net_rows)
    if row is None:
        lines = ["Net ArUco: --", "Pick: --", "Place: --"]
    else:
        _, _, net_aruco, pick_count, place_count, _ = row
        lines = [
            f"Net ArUco: {net_aruco:+d}",
            f"Pick:     {pick_count}",
            f"Place:    {place_count}",
            f"stride:   {FRAME_STRIDE}",
        ]

    # Draw marker boxes on sampled frames when detection cache is available
    if frame_idx in frame_detections:
        draw_marker_overlays(frame, frame_detections[frame_idx])

    draw_stacked_overlay(frame, lines)
    writer.write(frame)
    frame_idx += 1

cap.release()
writer.release()
print(
    f"{VIDEO_PATH} + {OUTPUT_CSV} -> {ANNOTATED_VIDEO}  "
    f"({frame_idx} frames, FRAME_STRIDE={FRAME_STRIDE})"
)

../hmm-testing/picklist_videos/picklist_191.MP4 + csv_outputs/picklist_191_aruco_net_timeseries.csv -> picklist_191_aruco_net_annotated.mp4  (2293 frames, FRAME_STRIDE=5)
